# Moirai 1.0 — Load


In [ ]:
# !pip install -r requirements/moirai.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"load"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
from uni2ts.distribution import StudentTOutput
from uni2ts.common.torch_util import set_seed
from preprocessing import load_data, describe_data, SITES, CONTEXT_LENGTH, PREDICTION_LENGTH
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=OUTPUT_QUANTILES
OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data=load_data(ROOT/"data")
print(describe_data(data))


## GEFCom2014-L


In [ ]:
# GEFCom2014-L (cleaned input; no extra cleaning)
from gefcom import load_gefcom_load, load_initial_and_tasks, evaluate_gefcom_load, LOAD_WEATHER
gef_load=load_gefcom_load(ROOT/"data"/"gefcom_load_data.csv")
gef_context, gef_tasks=load_initial_and_tasks(gef_load)
GEF_OUTPUT=ROOT/"outputs"/"predictions"; GEF_OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
GEF_VARIANT="with_covariates"; rolling=gef_context.copy(); frames=[]
for task_id,future in tqdm(gef_tasks):
    context=rolling.sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy(); past=LOAD_WEATHER if GEF_VARIANT=="with_covariates" else []; H=len(future)
    ds=PandasDataset.from_long_dataframe(context,target="target",item_id="item_id",timestamp="timestamp",freq="h",past_feat_dynamic_real=past,feat_dynamic_real=[])
    m=MoiraiForecast(module=MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-base"),prediction_length=H,context_length=CONTEXT_LENGTH,patch_size=8,num_samples=100,target_dim=1,feat_dynamic_real_dim=0,past_feat_dynamic_real_dim=len(past)); set_seed(42)
    fc=next(iter(m.create_predictor(batch_size=4).predict(ds))); samples=fc.samples
    d=pd.DataFrame({"item_id":future.item_id.to_numpy(),"timestamp":future.timestamp.to_numpy(),"task_id":task_id,"prediction":np.quantile(samples,0.5,axis=0)})
    for q in QUANTILES: d[str(q)]=np.quantile(samples,q,axis=0)
    frames.append(d); rolling=pd.concat([rolling,future],ignore_index=True)
gef_result=pd.concat(frames,ignore_index=True).merge(gef_load[["item_id","timestamp","target","task_id"]],on=["item_id","timestamp","task_id"],how="inner")
gef_result.to_csv(GEF_OUTPUT/f"moirai_gefcom_load_{GEF_VARIANT}.csv",index=False)
gef_summary=evaluate_gefcom_load(gef_result,QUANTILES); display(gef_summary[["task_id","raw_aql","normalized_aql"]])
gef_summary.to_csv(GEF_OUTPUT/f"moirai_gefcom_load_summary.csv",index=False)


## PG&E competition


In [ ]:
VARIANT="with_covariates"
FUTURE_COLS=data.covariates
PAST_COLS=[]
feat_cols=FUTURE_COLS if VARIANT=="with_covariates" else []
past_cols=PAST_COLS if VARIANT=="with_covariates" else []
model=MoiraiForecast(module=MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-base"),prediction_length=PREDICTION_LENGTH,context_length=CONTEXT_LENGTH,patch_size=8,num_samples=100,target_dim=1,feat_dynamic_real_dim=len(feat_cols),past_feat_dynamic_real_dim=len(past_cols))
set_seed(42); predictor=model.create_predictor(batch_size=4)


In [ ]:
rolling={s:data.context_by_site[s].copy() for s in data.context_by_site}
frames=[]
for i in tqdm(range(len(next(iter(data.daily_by_site.values()))))):
    day={s:data.daily_by_site[s][i][1].sort_values("timestamp").iloc[:PREDICTION_LENGTH].copy() for s in data.daily_by_site}
    for site,future in day.items():
        context=rolling[site].sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy(); full=pd.concat([context,future],ignore_index=True)
        ds=PandasDataset.from_long_dataframe(full,target="target",item_id="item_id",timestamp="timestamp",freq="h",past_feat_dynamic_real=past_cols,feat_dynamic_real=feat_cols)
        _,template=split(ds,offset=-PREDICTION_LENGTH); instances=template.generate_instances(prediction_length=PREDICTION_LENGTH,windows=1,distance=PREDICTION_LENGTH)
        for fc in predictor.predict(instances.input):
            qhat={q:np.quantile(fc.samples,q,axis=0) for q in MODEL_QUANTILES}; ts=pd.date_range(fc.start_date.to_timestamp(),periods=PREDICTION_LENGTH,freq="h")
            frame=pd.DataFrame({"item_id":fc.item_id,"timestamp":ts,"prediction":qhat[0.5]})
            for q in MODEL_QUANTILES: frame[str(q)]=qhat[q]
            frames.append(frame)
    for site in day: rolling[site]=pd.concat([rolling[site],day[site]],ignore_index=True).sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
result=pd.concat(frames,ignore_index=True); result["item_id"]=result["item_id"].astype(str); truth=data.test[["item_id","timestamp","target"]].copy(); truth["item_id"]=truth["item_id"].astype(str)
result=result.merge(truth,on=["item_id","timestamp"],how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR/f"moirai_{VARIANT}.csv",index=False)
print_metrics(result,QUANTILES,label=f"moirai_{VARIANT}")
